# 99 — Subir proyecto v2 a Google Drive

Ejecutá este notebook **una vez** después de cambiar código en `d10sformer-v2` local o en Git.

Destino: `MyDrive/d10sformer-v2` (id `1Xz1rbw8t8jF_6J5Ez-_vUb7MuPG69w-O`)

No sube `data/` (los datos quedan en `MyDrive/d10sformer`).

In [ ]:
!pip install -q google-api-python-client google-auth-oauthlib

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload
from pathlib import Path
import mimetypes

DRIVE = Path('/content/drive/MyDrive')
V2_FOLDER_ID = '1Xz1rbw8t8jF_6J5Ez-_vUb7MuPG69w-O'

# Destino en Drive (soporta carpeta anidada d10sformer-v2/d10sformer-v2)
for _cand in (DRIVE / 'd10sformer-v2' / 'd10sformer-v2', DRIVE / 'd10sformer-v2'):
    if (_cand / 'src').is_dir() or _cand.is_dir():
        V2_ROOT = _cand.resolve()
        break
else:
    V2_ROOT = (DRIVE / 'd10sformer-v2').resolve()

# Clonar / actualizar rama stg desde GitHub
REPO = 'https://github.com/jose-marin-db/udesa-nlp-futbol-D10Sformer.git'
BRANCH = 'stg'
CLONE_DIR = Path('/content/d10sformer-b')

if CLONE_DIR.exists() and (CLONE_DIR / '.git').exists():
    !cd /content/d10sformer-b && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !rm -rf /content/d10sformer-b
    !git clone -b {BRANCH} -q {REPO} /content/d10sformer-b

SOURCE = CLONE_DIR
assert (SOURCE / 'src' / 'data' / 'collator.py').is_file(), 'Clone falló — revisá REPO/BRANCH'
assert 'LabelMappedCollator' in (SOURCE / 'src' / 'data' / 'collator.py').read_text(), 'collator.py viejo en stg'

service = build('drive', 'v3')
print('BRANCH  =', BRANCH)
print('SOURCE  =', SOURCE)
print('V2_ROOT =', V2_ROOT, '(subida a esta carpeta)')

In [ ]:
import shutil

# Copia directa al path montado en Drive (respeta d10sformer-v2/d10sformer-v2)
SKIP_TOP = {'.git', '__pycache__', '.ipynb_checkpoints', 'checkpoints', 'data', 'reports'}
UPLOAD_EXT = {'.py', '.ipynb', '.md', '.yaml', '.txt', '.json'}

# 1) src/ completo (lo más importante)
shutil.copytree(SOURCE / 'src', V2_ROOT / 'src', dirs_exist_ok=True)
print(f'✓ src/ → {V2_ROOT / "src"}')

# 2) notebooks + configs + scripts + docs sueltos
count = 0
for path in sorted(SOURCE.rglob('*')):
    if not path.is_file() or path.suffix not in UPLOAD_EXT:
        continue
    rel = path.relative_to(SOURCE)
    if not rel.parts or rel.parts[0] in SKIP_TOP:
        continue
    if rel.parts[0] == 'src':
        continue
    dest = V2_ROOT / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, dest)
    count += 1
    print(f'  {rel}')

collator = V2_ROOT / 'src' / 'data' / 'collator.py'
ok = 'class LabelMappedCollator' in collator.read_text(encoding='utf-8')
pad_ok = 'def pad_id' in (V2_ROOT / 'src' / 'data' / 'vocabulary.py').read_text(encoding='utf-8')
print(f'\n✓ {count} archivos extra + src/')
print(f'  LabelMappedCollator: {ok}  |  vocab.pad_id: {pad_ok}')
print(f'  Destino final: {V2_ROOT}')